# Làm sạch dữ liệu bằng Pandas

Chạy từ trên xuống bằng kernel container audit. Quy tắc chi tiết nằm trong docs/data_cleaning.md.

Xóa dòng thiếu children, ADR âm, ADR trên 1000 và bản sao dư của dòng trùng hoàn toàn. Thống nhất country: CN thành CHN, TMP thành TLS. Điền Unknown cho country, agent, company. Giữ dòng có tổng khách bằng 0, tổng đêm bằng 0 hoặc ADR bằng 0. CSV kết quả gồm 32 cột nguồn và cột arrival_date; báo cáo JSON lưu kết quả kiểm tra.


Hai thời gian được đo khi chạy **Run All** liên tục:

- **Toàn quy trình:** đọc và chuẩn hóa đầu vào, làm sạch, kiểm tra rồi ghi CSV lên HDFS.
- **Chỉ xử lý:** làm sạch và kiểm tra sau khi dữ liệu đã được đọc/chuẩn hóa, không tính khởi động engine hay đọc/ghi HDFS.

Thời gian xử lý được tính qua các cell; không dừng lâu giữa các cell khi benchmark. Pandas và PySpark dùng cách đọc/ghi HDFS khác nhau, nên thời gian toàn quy trình phản ánh cả connector.


## 1. Cấu hình

In [1]:
import json
import os
import tempfile
import time
from datetime import datetime, date
import re
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

INPUT_FILE = os.environ.get(
    "AUDIT_HDFS_FILE",
    "/data/hotel_bookings/input/hotel_bookings_synthetic_200MB.csv",
)
WEBHDFS_BASE = "http://namenode:9870/webhdfs/v1"
INPUT_URL = f"{WEBHDFS_BASE}{INPUT_FILE}?op=OPEN&user.name=root"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_FILE = f"/data/hotel_bookings/cleaned/pandas/{RUN_ID}/hotel_bookings.csv"

NUMERIC_COLUMNS = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
]
CATEGORY_COLUMNS = ["country", "agent", "company"]
COUNTRY_CODE_MAP = {"CN": "CHN", "TMP": "TLS"}
INTEGER_COLUMNS = [column for column in NUMERIC_COLUMNS if column != "adr"]
BINARY_COLUMNS = ["is_canceled", "is_repeated_guest"]
VALID_RANGES = {
    "arrival_date_year": (1, 9999),
    "arrival_date_week_number": (1, 53),
    "arrival_date_day_of_month": (1, 31),
}
MONTHS = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}
CANONICAL_VALUES = {
    "hotel": ["City Hotel", "Resort Hotel"],
    "reservation_status": ["Canceled", "Check-Out", "No-Show"],
    "arrival_date_month": list(MONTHS),
}
SOURCE_COLUMNS = [
    "hotel", "is_canceled", "lead_time", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies", "meal", "country",
    "market_segment", "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled", "reserved_room_type",
    "assigned_room_type", "booking_changes", "deposit_type", "agent", "company",
    "days_in_waiting_list", "customer_type", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "reservation_status", "reservation_status_date",
]
MAX_SAFE_INTEGER = 2**53 - 1
RULES_VERSION = "hotel-clean-v5"
BENCHMARK_VERSION = "clean-benchmark-v1"
MISSING_WORDS = ["", "nan", "null", "none", "na", "n/a"]
REPORT_DIR = Path("/workspace/reports")


## 2. Đọc dữ liệu và chuẩn hóa null

Kiểm tra đúng 32 cột đầu vào. Chuẩn hóa khoảng trắng, chuỗi thiếu và mã quốc gia (CN → CHN, TMP → TLS). Sau đó so sánh trùng trên toàn bộ 32 cột; giữ một dòng, xóa các bản sao còn lại.


In [2]:
pipeline_started_at = time.perf_counter()
raw = pd.read_csv(INPUT_URL, dtype="string", keep_default_na=False, low_memory=False)
raw.columns = raw.columns.str.strip()
if raw.columns.duplicated().any() or set(raw.columns) != set(SOURCE_COLUMNS):
    raise ValueError("CSV phải có đúng 32 cột gốc, không chứa cột trùng hoặc cột kết quả.")

cleaned = raw.copy()
for column in SOURCE_COLUMNS:
    values = cleaned[column].str.strip()
    cleaned[column] = values.mask(values.str.lower().isin(MISSING_WORDS), pd.NA)

source_read_seconds = time.perf_counter() - pipeline_started_at
processing_started_at = time.perf_counter()

cleaned["country"] = cleaned["country"].str.upper().replace(COUNTRY_CODE_MAP)

duplicate_mask = cleaned.duplicated(subset=SOURCE_COLUMNS, keep="first")
duplicate_rows_removed = int(duplicate_mask.sum())
cleaned = cleaned.loc[~duplicate_mask].copy()
filled_missing = {
    column: int(cleaned[column].isna().sum())
    for column in CATEGORY_COLUMNS
}
cleaned[CATEGORY_COLUMNS] = cleaned[CATEGORY_COLUMNS].fillna("Unknown")
print(f"Số dòng đầu vào: {len(raw):,}")
print(f"Bản sao trùng đã xóa: {duplicate_rows_removed:,}")


Số dòng đầu vào: 1,383,291
Bản sao trùng đã xóa: 53,675


## 3. Làm sạch số và xóa dòng theo quy tắc

Giá trị số không hợp lệ được chuyển thành null. Sau đó xóa dòng thiếu `children`, có ADR âm hoặc ADR trên 1000. ADR không đọc được được giữ ở dạng null.

In [3]:
for column in NUMERIC_COLUMNS:
    source = cleaned[column]
    values = pd.to_numeric(source, errors="coerce").astype("Float64")
    invalid = (source.notna() & values.isna()) | values.isin([float("inf"), -float("inf")])
    values = values.mask(invalid.fillna(False))

    if column == "adr":
        cleaned[column] = values
        continue

    invalid_count = (values.lt(0) | values.mod(1).ne(0) | values.gt(MAX_SAFE_INTEGER)).fillna(False)
    invalid_binary = pd.Series(False, index=cleaned.index)
    invalid_range = pd.Series(False, index=cleaned.index)

    if column in BINARY_COLUMNS:
        invalid_binary = values.notna() & ~values.isin([0, 1])
    if column in VALID_RANGES:
        lower, upper = VALID_RANGES[column]
        invalid_range = (values.lt(lower) | values.gt(upper)).fillna(False)

    cleaned[column] = values.mask(
        invalid_count | invalid_binary | invalid_range
    ).astype("Int64")

missing_children = cleaned["children"].isna()
removed_children_missing = int(missing_children.sum())
cleaned = cleaned.loc[~missing_children].copy()
negative_adr = cleaned["adr"].lt(0).fillna(False)
removed_negative_adr = int(negative_adr.sum())
cleaned = cleaned.loc[~negative_adr].copy()
adr_above_1000 = cleaned["adr"].gt(1000).fillna(False)
removed_adr_above_1000 = int(adr_above_1000.sum())
cleaned = cleaned.loc[~adr_above_1000].copy()

## 4. Chuẩn hóa mã phân loại

Chuẩn hóa chữ hoa/thường ở hotel, trạng thái, tháng và country. Mã `agent`/`company` dạng `9.00` được chuẩn hóa thành `9`; các giá trị khác được giữ nguyên.

In [4]:
for column, allowed_values in CANONICAL_VALUES.items():
    lookup = {value.lower(): value for value in allowed_values}
    canonical = cleaned[column].str.lower().map(lookup).astype("string")
    cleaned[column] = canonical.fillna(cleaned[column])

country = cleaned["country"]
cleaned["country"] = country.str.upper().mask(country.str.lower().eq("unknown"), "Unknown")
for column in ["agent", "company"]:
    values = cleaned[column].mask(cleaned[column].str.lower().eq("unknown"), "Unknown")
    cleaned[column] = values.str.replace(r"^([0-9]+)\.0+$", r"\1", regex=True)

## 5. Chuẩn hóa ngày

Tạo `arrival_date` theo định dạng ISO. Ngày không hợp lệ được chuyển thành null; các giá trị ngày còn lại được chuẩn hóa.

In [5]:
def parse_iso_dates(values):
    def parse_one(text):
        try:
            if re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", text) is None:
                return None
            return date.fromisoformat(text).isoformat()
        except ValueError:
            return None

    lookup = {text: parse_one(text) for text in values.dropna().unique()}
    return values.map(lookup).astype("string")

month_number = cleaned["arrival_date_month"].map(MONTHS).astype("Int64")
arrival_text = (
    cleaned["arrival_date_year"].astype("string").str.zfill(4) + "-"
    + month_number.astype("string").str.zfill(2) + "-"
    + cleaned["arrival_date_day_of_month"].astype("string").str.zfill(2)
)
cleaned["arrival_date"] = parse_iso_dates(arrival_text)
cleaned["reservation_status_date"] = parse_iso_dates(cleaned["reservation_status_date"])

## 6. Kiểm tra và tổng hợp

Tổng khách bằng 0, tổng đêm bằng 0 và ADR bằng 0 được giữ nguyên. ADR trên 1000 đã bị xóa ở bước trước.

## 7. Kiểm tra kết quả trước khi ghi

Notebook kiểm tra các quy tắc xóa, kiểu số, giá trị phân loại và số null còn lại. Nếu kiểm tra thất bại, notebook dừng trước khi ghi dữ liệu.

In [6]:
remaining_missing = {
    column: int(cleaned[column].isna().sum())
    for column in SOURCE_COLUMNS + ["arrival_date"]
}
post_clean_checks = {
    "row_count_difference": abs(
        (len(raw) - duplicate_rows_removed - removed_children_missing - removed_negative_adr
         - removed_adr_above_1000)
        - len(cleaned)
    ),
    "children_missing": int(cleaned["children"].isna().sum()),
    "nonfinite_numeric": int(cleaned[NUMERIC_COLUMNS].isin([float("inf"), -float("inf")]).sum().sum()),
    "invalid_counts": sum(
        int((cleaned[column].lt(0) | cleaned[column].mod(1).ne(0)).fillna(False).sum())
        for column in INTEGER_COLUMNS
    ),
    "invalid_binary": sum(
        int((cleaned[column].notna() & ~cleaned[column].isin([0, 1])).sum())
        for column in BINARY_COLUMNS
    ),
    "invalid_range": sum(
        int((cleaned[column].lt(lower) | cleaned[column].gt(upper)).fillna(False).sum())
        for column, (lower, upper) in VALID_RANGES.items()
    ),
    "negative_adr": int(cleaned["adr"].lt(0).fillna(False).sum()),
    "adr_above_1000": int(cleaned["adr"].gt(1000).fillna(False).sum()),
    "null_categories": int(cleaned[CATEGORY_COLUMNS].isna().sum().sum()),
    "country_aliases_remaining": int(cleaned["country"].isin(list(COUNTRY_CODE_MAP)).sum()),
}
processing_seconds = time.perf_counter() - processing_started_at
display(pd.DataFrame(post_clean_checks.items(), columns=["check", "remaining_errors"]))
if any(post_clean_checks.values()):
    raise ValueError("Kết quả chưa đạt kiểm tra; dừng trước khi ghi HDFS.")

display(pd.DataFrame(remaining_missing.items(), columns=["column", "remaining_nulls"]))

,check,remaining_errors
0,row_count_difference,0
1,children_missing,0
2,nonfinite_numeric,0
3,invalid_counts,0
4,invalid_binary,0
5,invalid_range,0
6,negative_adr,0
7,adr_above_1000,0
8,null_categories,0
9,country_aliases_remaining,0


,column,remaining_nulls
0,hotel,0
1,is_canceled,0
2,lead_time,0
3,arrival_date_year,0
4,arrival_date_month,0
5,arrival_date_week_number,0
6,arrival_date_day_of_month,0
7,stays_in_weekend_nights,0
8,stays_in_week_nights,0
9,adults,0


## 8. Ghi CSV mới lên HDFS

Pandas tạo CSV tạm trong container rồi upload qua WebHDFS. File tạm được dọn sau khi upload; CSV kết quả nằm trên HDFS. Báo cáo JSON nằm ở reports/ trên máy local. Mỗi lần chạy lại toàn bộ notebook sẽ tạo thư mục kết quả mới.

In [7]:
if any(post_clean_checks.values()):
    raise ValueError("Kết quả chưa đạt kiểm tra; dừng trước khi ghi HDFS.")

def upload_to_hdfs(local_file, output_file):
    parent_folder = output_file.rsplit("/", 1)[0]
    with requests.Session() as client:
        client.trust_env = False
        folder = client.put(
            f"{WEBHDFS_BASE}{parent_folder}",
            params={"op": "MKDIRS", "user.name": "root"},
            timeout=30,
        )
        folder.raise_for_status()
        create = client.put(
            f"{WEBHDFS_BASE}{output_file}",
            params={"op": "CREATE", "overwrite": "false", "user.name": "root"},
            allow_redirects=False,
            timeout=30,
        )
        create.raise_for_status()
        if create.status_code != 307:
            raise RuntimeError(f"WebHDFS CREATE trả về {create.status_code}, cần 307.")

        with open(local_file, "rb") as stream:
            uploaded = client.put(
                create.headers["Location"],
                data=stream,
                headers={"Content-Type": "application/octet-stream"},
                timeout=(30, 600),
            )
        uploaded.raise_for_status()

write_started_at = time.perf_counter()
with tempfile.TemporaryDirectory() as temporary_folder:
    temporary_csv = Path(temporary_folder) / "hotel_bookings.csv"
    cleaned[SOURCE_COLUMNS + ["arrival_date"]].to_csv(temporary_csv, index=False, na_rep="")
    upload_to_hdfs(temporary_csv, OUTPUT_FILE)
hdfs_write_seconds = time.perf_counter() - write_started_at
pipeline_seconds = time.perf_counter() - pipeline_started_at
total_seconds = pipeline_seconds

summary = {
    "engine": "pandas",
    "benchmark_version": BENCHMARK_VERSION,
    "source": INPUT_FILE,
    "output": OUTPUT_FILE,
    "input_rows": int(len(raw)),
    "output_rows": int(len(cleaned)),
    "filled_missing": filled_missing,
    "rules_version": RULES_VERSION,
    "post_clean_checks": post_clean_checks,
    "remaining_missing": remaining_missing,
    "removed_rows": {
        "exact_duplicate_extra": duplicate_rows_removed,
        "children_missing": removed_children_missing,
        "negative_adr": removed_negative_adr,
        "adr_above_1000": removed_adr_above_1000,
        "total": duplicate_rows_removed + removed_children_missing + removed_negative_adr + removed_adr_above_1000,
    },
    "duplicate_policy": "drop_extra_keep_first",
    "source_read_seconds": round(source_read_seconds, 3),
    "processing_seconds": round(processing_seconds, 3),
    "hdfs_write_seconds": round(hdfs_write_seconds, 3),
    "read_clean_write_seconds": round(pipeline_seconds, 3),
    "pipeline_seconds": round(pipeline_seconds, 3),
    "spark_startup_seconds": 0.0,
    "total_seconds": round(total_seconds, 3),
}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "clean_pandas.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Đã ghi: hdfs://namenode:9000{OUTPUT_FILE}")
print(f"Đọc và chuẩn hóa: {source_read_seconds:.2f} giây")
print(f"Làm sạch và kiểm tra: {processing_seconds:.2f} giây")
print(f"Ghi HDFS: {hdfs_write_seconds:.2f} giây")
print(f"Toàn quy trình: {total_seconds:.2f} giây")


Đã ghi: hdfs://namenode:9000/data/hotel_bookings/cleaned/pandas/20261022_093034_748648/hotel_bookings.csv
Đọc và chuẩn hóa: 11.96 giây
Làm sạch và kiểm tra: 17.14 giây
Ghi HDFS: 7.51 giây
Toàn quy trình: 36.62 giây
